In [ ]:
import src.travel_agent.config  # loads .env and sets API key
from agents import Runner, trace
from src.travel_agent.agents.intake_agent import intake_agent
from src.travel_agent.agents.visa_agent import visa_agent
from src.travel_agent.agents.activities_agent import activities_agent

## 1. Intake: multi-turn chat


In [9]:
history = []

async def say(msg):
    global history
    result = await Runner.run(intake_agent, history + [{'role': 'user', 'content': msg}])
    history = result.to_input_list()
    out = result.final_output
    print('missing:', out.missing_fields)
    print('assistant:', out.next_question)
    return out

out = await say('I want to visit Japan with 2 friends on Oct 26 for 7 days, budget 6000 USD')

missing: ['trip_type', 'passport_country', 'origin', 'interests']
assistant: What country is your passport from?


In [11]:
out = await say('I hold a French passport, flying from Miami. Food and temples for vacation.')

missing: []
assistant: None


In [12]:
trip = out.trip
print(trip.model_dump_json(indent=2))
print('complete:', out.is_complete)

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": "2026-10-26",
  "end_date": "2026-11-01",
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "Food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}
complete: True


## 2. Visa agent


In [13]:
with trace(""):
    r = await Runner.run(visa_agent, f'Passport: {trip.passport_country}. Destination: {trip.destination}.')
print(r.final_output.model_dump_json(indent=2))

{
  "passport": "FRA",
  "destination": "JPN",
  "requirement": "visa_free",
  "allowed_days": 90,
  "summary": "A passport from France allows visa-free travel to Japan for up to 90 days. No visa is required for stays within this period.",
  "source": "orizn_api"
}


## 3. Activities agent


In [14]:
prompt = (
    f'Destination: {trip.destination}. Dates: {trip.start_date} to {trip.end_date}. '
    f'Travelers: {trip.travelers}. Budget USD: {trip.budget_usd}. '
    f'Interests: {trip.interests}. Trip type: {trip.trip_type}.'
)
with trace('Day4 activities'):
    r = await Runner.run(activities_agent, prompt)
plan = r.final_output
print(plan.weather_summary)
for a in plan.activities:
    print('-', a.name, '|', a.category, '|', a.estimated_cost_usd)

Typical seasonal weather for late October to early November in Japan is generally mild and pleasant, with less rain, suitable for outdoor activities and temple visits.
- Kyoto Temple and Historical Sites Tour | temples | 30.0
- Karez Curry Restaurant (Kyoto) | food | 15.0
- Nouvel Bar Kyoto (Kyoto) | food | 40.0
- Bungalow Terramachi (Kyoto) | food | 25.0
- Nara Temple Excursion: Visit Saikyo-ji and other temples | temples | 20.0
- Dining at Shokudo Kanbashi (Nara) | food | 20.0
- Okonomiyaki Manpuku (Nara) | food | 15.0
